In [1]:
import pandas as pd 
import os 
import re
import numpy as np 
import warnings
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
pd.options.display.max_columns = 100
pd.options.display.max_rows = 60


# 0. Preface
This notebook will preprocess the eICU dataset into a features matrix that you can use for machine learning. 

# 1. Introduction: the eICU Dataset

The eICU Collaborative Research Database is a large, real-world ICU dataset containing detailed information about patient care across many hospitals in the United States. It includes vital signs, laboratory measurements, diagnoses, treatments, and severity scores, all collected during ICU stays.

Unlike a typical “single table” dataset, eICU is stored as a **relational database**. This means the data is split across multiple tables, each representing a different aspect of patient care. These tables are linked together using shared identifiers such as:

* `patientunitstayid` → a single ICU visit (also called a “unit”) within a hospitalization
* `patienthealthsystemstayid` → a hospitalization visit
* `hospitalid` → the hospital itself (there are many hospitals in this dataset)

Relational databases are common, especially in healthcare. The key idea is that for each patient’s ICU visit (`patientunitstayid`), their data is spread across many tables, often with multiple rows per table. For example, in `lab.csv.gz`, a patient may have many lab results recorded during a single ICU stay.

---

### Table outline

While there are many tables in the eICU dataset, we will focus on a few core ones:

* `patient.csv.gz` → patient demographics and hospital-level information  
* `diagnosis.csv.gz` → diagnoses (ICD codes) recorded during the ICU stay  
* `admissionDx.csv.gz` → admission diagnoses in unstructured text (right before ICU admission)  
* `lab.csv.gz` → lab test results (e.g., creatinine, WBC)  
* `vitalPeriodic.csv.gz` and `vitalAperiodic.csv.gz` → vital signs (e.g., heart rate, blood pressure) recorded over time  

You are strongly encouraged to explore the full table descriptions here:  
https://eicu-crd.mit.edu/eicutables/admissiondrug/

Helpful example notebooks are available here:  
https://github.com/MIT-LCP/eicu-code/tree/main/notebooks

---

### Creating the cohort

This dataset can be processed into features \(X\) in many different ways. In this project, we will use the following setup:

* We consider patients who were in the hospital for at least 48 hours  
* We ensure that features \(X\) occur before the outcome \(Y\) (so the model is actually predictive)  
* Features \(X\): diagnoses, vitals, and labs recorded during hours 0–24 of the hospital stay (the first day)  
* Outcomes \(Y\): events observed during hours 24–48 (the second day), e.g., patient mortality  

---

### A note on timing

Timing in this dataset can be a bit confusing.

For each hospitalization (`patienthealthsystemstayid`), a patient may have multiple ICU stays (`patientunitstayid`). In this project, we focus only on ICU data within the first 48 hours of the initial hospitalization.

To determine when events (labs, diagnoses, vitals, etc.) occur, we use the *offset* columns in each table, which record the number of minutes from a reference time. These offsets allow us to align events across tables and restrict them to the time windows defined above.

We only consider features \(X\) recorded at any ICU visit within the first 24 hours of hospitalization; for outcomes \(Y\) we can take any values observed during any ICU visit within 24-48 hours of hospitalization.

In [2]:
# Flag as true if you are working witgh the eICU demo dataset; False if you have the full dataset 
demo = True 

# Make sure you are in the directory where you downloaded the data
if demo:
    data_dir = '../data/eicu-collaborative-research-database-demo-2.0.1/' #TODO: make sure this points to your directory containing the demo dataset
else:
    data_dir = '../data/physionet.org/files/eicu-crd/2.0/' #TODO: make sure this points to your directory containing the full dataset

assert os.path.exists(data_dir), f"Data directory {data_dir} does not exist. Please check that you've downloaded the data to the correct path name." 
needed_files = ['diagnosis.csv.gz', 'vitalAperiodic.csv.gz', 
                'admissionDx.csv.gz', 'hospital.csv.gz', 'vitalPeriodic.csv.gz', 
                'patient.csv.gz', 'treatment.csv.gz', 'apacheApsVar.csv.gz', 
                'lab.csv.gz', 'apachePredVar.csv.gz', 
                'apachePatientResult.csv.gz']
assert np.setdiff1d(needed_files, os.listdir(data_dir)).shape[0] == 0 # Folder should contain all the needed files. If not, check that you've downloaded the data correctly.


# 2. Load & clean the data 
We will create a table / matrix / feature set `patient_agg` where each row is a data point and the columns are all possible predictive features X or labels Y we might care about. That way, when we train our ML models, we will have all our data easily accessible in one place! 

### Patient table

The `patient.csv.gz` table serves as the "anchor" table for the dataset. Each row corresponds to a single ICU stay (`patientunitstayid`) and includes basic information about the patient and their hospitalization.

This table contains:
* Demographics (e.g., age, gender)
* Admission and discharge information
* Hospital identifiers (e.g., `hospitalid`)
* Timing information (e.g., length of stay, admission offsets)

In practice, you will almost always start with the `patient` table when defining a cohort, and then join other tables (labs, vitals, diagnoses, etc.) using `patientunitstayid` to gather additional features.

In [3]:
patient = pd.read_csv(f'{data_dir}/patient.csv.gz')
patient

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,admissionheight,hospitaladmittime24,hospitaladmitoffset,hospitaladmitsource,hospitaldischargeyear,hospitaldischargetime24,hospitaldischargeoffset,hospitaldischargelocation,hospitaldischargestatus,unittype,unitadmittime24,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid
0,141764,129391,Female,87,Caucasian,59,91,NaN,157.5,23:36:00,-2258,NaN,2015,19:20:00,366,Home,Alive,Med-Surg ICU,13:14:00,ICU to SDU,2,stepdown/other,NaN,NaN,18:58:00,344,Home,Alive,002-1039
1,141765,129391,Female,87,Caucasian,59,91,"Rhythm disturbance (atrial, supraventricular)",157.5,23:36:00,-8,NaN,2015,19:20:00,2616,Home,Alive,Med-Surg ICU,23:44:00,Emergency Department,1,admit,46.5,45.0,13:14:00,2250,Step-Down Unit (SDU),Alive,002-1039
2,143870,131022,Male,76,Caucasian,68,103,"Endarterectomy, carotid",167.0,20:46:00,-1,Operating Room,2014,17:05:00,1218,Home,Alive,SICU,20:47:00,Operating Room,1,admit,77.5,79.4,10:00:00,793,Floor,Alive,002-12289
3,144815,131736,Female,34,Caucasian,56,82,"Overdose, other toxin, poison or drug",172.7,01:44:00,-23,Emergency Department,2015,21:05:00,1138,Other Hospital,Alive,Med-Surg ICU,02:07:00,Emergency Department,1,admit,60.3,60.7,20:48:00,1121,Other External,Alive,002-1116
4,145427,132209,Male,61,Caucasian,68,103,"GI perforation/rupture, surgery for",177.8,23:48:00,-10,Emergency Department,2014,15:41:00,5263,Home,Alive,SICU,23:58:00,Operating Room,1,admit,91.7,93.1,22:47:00,1369,Floor,Alive,002-12243
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2515,3351763,2741766,Female,62,Caucasian,459,1108,"Fistula/abscess, surgery for (not inflammatory...",165.1,16:08:00,-68242,Operating Room,2014,16:24:00,36894,Home,Alive,Med-Surg ICU,01:30:00,Operating Room,1,admit,134.5,133.3,19:24:00,5394,Step-Down Unit (SDU),Alive,035-10391
2516,3352230,2742186,Male,41,African American,458,1107,"CABG alone, coronary artery bypass grafting",177.8,21:21:00,-1512,Direct Admit,2014,18:35:00,18482,Home,Alive,CTICU,22:33:00,Operating Room,2,transfer,127.0,128.5,21:34:00,4261,Telemetry,Alive,035-10089
2517,3352231,2742186,Male,41,African American,458,1104,"Infarction, acute myocardial (MI)",177.8,21:21:00,-136,Direct Admit,2014,18:35:00,19858,Home,Alive,Cardiac ICU,23:37:00,Direct Admit,1,admit,127.0,135.2,22:26:00,1369,Other ICU,Alive,035-10089
2518,3352333,2742269,Male,72,Caucasian,458,1111,GI obstruction,177.8,20:00:00,0,Other Hospital,2014,16:59:00,5579,NaN,Alive,SICU,20:00:00,Other Hospital,1,admit,68.3,66.5,17:26:00,4166,Floor,Alive,035-10041


In [4]:
# Total number of unique patients (using uniquepid, which identifies a person across stays)
N = patient.uniquepid.nunique()

# Define time windows (in hours)
hours_mortality = 48   # outcome window: we will look at mortality within 48 hours
hours_for_data = 24    # feature window: we only use data from the first 24 hours

# -----------------------------
# Step 1: Filter out short hospital visits < 24 hours 
# -----------------------------
# hospitaldischargeoffset is in minutes (time from hospital admission to discharge)
# We require patients to stay at least 24 hours so that we have enough data to build features
patient = patient[patient.hospitaldischargeoffset >= (hours_for_data * 60)]

print(f"Removed {N - patient.uniquepid.nunique()} patients who were discharged before {hours_for_data} hours")

Removed 200 patients who were discharged before 24 hours


In [5]:
# Update patient count after filtering
N = patient.uniquepid.nunique()

# -----------------------------
# Step 2: Ensure valid 48h window
# -----------------------------
# hospitaladmitoffset is typically negative (minutes BEFORE ICU admission)
# -hospitaladmitoffset converts it into "minutes since hospital admission"
# We want ICU stays that occur within the first 48 hours of hospitalization
invalid = -patient.hospitaladmitoffset > hours_mortality * 60

print(f"Removing {invalid.sum()} ICU visits that start after {hours_mortality} hours = "
      f"Removing {N - patient.uniquepid.nunique()} patients who have no remaining data (usually 0)")

# Keep only valid ICU stays
patient = patient[~invalid]

Removing 378 ICU visits that start after 48 hours = Removing 0 patients who have no remaining data (usually 0)


In [6]:
# -----------------------------
# Step 3: Define one possible outcome Y (mortality wihtin 48 hours)
# -----------------------------
# unitdischargestatus == 'Expired' means the patient died during that ICU stay
patient[f'mortality_at{hours_mortality}h'] = (patient.unitdischargestatus == 'Expired').astype(int)

# A patient can have multiple ICU stays within one hospitalization
# We aggregate to the hospitalization level:
# if the patient died in ANY ICU stay → mark the whole hospitalization as mortality = 1
patient[f'mortality_at{hours_mortality}h'] = (
    patient.groupby('patienthealthsystemstayid')[f'mortality_at{hours_mortality}h']
    .transform('max')
)
patient 

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,admissionheight,hospitaladmittime24,hospitaladmitoffset,hospitaladmitsource,hospitaldischargeyear,hospitaldischargetime24,hospitaldischargeoffset,hospitaldischargelocation,hospitaldischargestatus,unittype,unitadmittime24,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid,mortality_at48h
1,141765,129391,Female,87,Caucasian,59,91,"Rhythm disturbance (atrial, supraventricular)",157.5,23:36:00,-8,NaN,2015,19:20:00,2616,Home,Alive,Med-Surg ICU,23:44:00,Emergency Department,1,admit,46.5,45.0,13:14:00,2250,Step-Down Unit (SDU),Alive,002-1039,0
4,145427,132209,Male,61,Caucasian,68,103,"GI perforation/rupture, surgery for",177.8,23:48:00,-10,Emergency Department,2014,15:41:00,5263,Home,Alive,SICU,23:58:00,Operating Room,1,admit,91.7,93.1,22:47:00,1369,Floor,Alive,002-12243,0
7,147784,134042,Female,60,Hispanic,67,109,Coma/change in level of consciousness (for hep...,154.9,05:06:00,0,NaN,2015,23:08:00,3962,Home,Alive,Med-Surg ICU,05:06:00,Emergency Department,1,admit,95.6,97.6,20:47:00,2381,Floor,Alive,002-10424,0
11,149713,135537,Female,> 89,Caucasian,60,83,"Infarction, acute myocardial (MI)",157.5,19:05:53,-8,Emergency Department,2015,20:50:00,2977,Skilled Nursing Facility,Alive,Med-Surg ICU,19:13:00,Emergency Department,1,admit,58.5,58.6,14:52:00,1179,Step-Down Unit (SDU),Alive,002-11569,0
12,149714,135537,Female,> 89,Caucasian,60,83,NaN,157.5,19:05:53,-1187,Emergency Department,2015,20:50:00,1798,Skilled Nursing Facility,Alive,Med-Surg ICU,14:52:00,ICU to SDU,2,stepdown/other,NaN,57.1,02:15:00,683,Floor,Alive,002-11569,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2514,3351297,2741350,Male,52,African American,458,1109,"Angina, stable (asymp or stable pattern of sym...",159.0,18:10:00,0,Direct Admit,2015,18:15:00,15845,Home,Alive,Cardiac ICU,18:10:00,Direct Admit,1,admit,77.3,75.0,19:00:00,1490,Telemetry,Alive,035-10059,0
2516,3352230,2742186,Male,41,African American,458,1107,"CABG alone, coronary artery bypass grafting",177.8,21:21:00,-1512,Direct Admit,2014,18:35:00,18482,Home,Alive,CTICU,22:33:00,Operating Room,2,transfer,127.0,128.5,21:34:00,4261,Telemetry,Alive,035-10089,0
2517,3352231,2742186,Male,41,African American,458,1104,"Infarction, acute myocardial (MI)",177.8,21:21:00,-136,Direct Admit,2014,18:35:00,19858,Home,Alive,Cardiac ICU,23:37:00,Direct Admit,1,admit,127.0,135.2,22:26:00,1369,Other ICU,Alive,035-10089,0
2518,3352333,2742269,Male,72,Caucasian,458,1111,GI obstruction,177.8,20:00:00,0,Other Hospital,2014,16:59:00,5579,NaN,Alive,SICU,20:00:00,Other Hospital,1,admit,68.3,66.5,17:26:00,4166,Floor,Alive,035-10041,0


In [7]:
# -----------------------------
# Step 4: Encode admission diagnosis
# -----------------------------
# apacheadmissiondx is a categorical variable (text labels of admission diagnosis)
# that are made as a reason for why the patient was admitted to the ICU
# Convert it into one-hot encoded features (one column per diagnosis)
admissiondx_ohe = (
    pd.get_dummies(patient.apacheadmissiondx, dummy_na=True)
    .astype(int)
    .add_prefix('admissiondx_')
)

# Inspect most common diagnoses (by prevalence)
print(admissiondx_ohe.mean().sort_values(ascending=False)[:40])

# Replace original column with one-hot encoded features
patient = pd.concat(
    [patient.drop(columns='apacheadmissiondx'), admissiondx_ohe],
    axis=1
)

# Final processed patient table
patient

admissiondx_nan                                                                                                                             0.071935
admissiondx_Sepsis, pulmonary                                                                                                               0.053406
admissiondx_Diabetic ketoacidosis                                                                                                           0.052316
admissiondx_CHF, congestive heart failure                                                                                                   0.045232
admissiondx_Pneumonia, bacterial                                                                                                            0.031608
admissiondx_Infarction, acute myocardial (MI)                                                                                               0.031063
admissiondx_Sepsis, renal/UTI (including bladder)                                                         

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,admissionheight,hospitaladmittime24,hospitaladmitoffset,hospitaladmitsource,hospitaldischargeyear,hospitaldischargetime24,hospitaldischargeoffset,hospitaldischargelocation,hospitaldischargestatus,unittype,unitadmittime24,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid,mortality_at48h,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Anaphylaxis,admissiondx_Anemia,"admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep",...,admissiondx_Peritonitis,"admissiondx_Pneumonia, aspiration","admissiondx_Pneumonia, bacterial","admissiondx_Pneumonia, other","admissiondx_Pneumonia, viral",admissiondx_Pneumothorax,admissiondx_Pre-eclampsia/eclampsia,admissiondx_Renal bleeding,"admissiondx_Renal failure, acute",admissiondx_Renal infection/abscess,admissiondx_Renal obstruction,"admissiondx_Respiratory - medical, other","admissiondx_Respiratory surgery, other","admissiondx_Restrictive lung disease (i.e., Sarcoidosis, pulmonary fibrosis)","admissiondx_Rhythm disturbance (atrial, supraventricular)",admissiondx_Rhythm disturbance (conduction defect),admissiondx_Rhythm disturbance (ventricular),admissiondx_Seizures (primary-no structural brain disease),"admissiondx_Sepsis, GI","admissiondx_Sepsis, cutaneous/soft tissue","admissiondx_Sepsis, other","admissiondx_Sepsis, pulmonary","admissiondx_Sepsis, renal/UTI (including bladder)","admissiondx_Sepsis, unknown","admissiondx_Shock, cardiogenic",admissiondx_Sickle cell crisis,"admissiondx_Skin surgery, other","admissiondx_Spinal cord surgery, other",admissiondx_Spinal/extremity trauma,admissiondx_Spinal/multiple trauma,admissiondx_Splenectomy,admissiondx_Stereotactic procedure,admissiondx_Subarachnoid hemorrhage/intracranial aneurysm,"admissiondx_Subarachnoid hemorrhage/intracranial aneurysm, surgery for","admissiondx_TURP, transurethral prostate resection for benign prostatic hypertrophy","admissiondx_Tamponade, pericardial","admissiondx_Thoracotomy for benign tumor (i.e. mediastinal chest wall mass, thymectomy)",admissiondx_Thoracotomy for lung cancer,admissiondx_Thoracotomy for pleural disease,admissiondx_Thrombectomy (with general anesthesia),"admissiondx_Thrombosis, vascular (deep vein)","admissiondx_Thrombus, arterial","admissiondx_Toxicity, drug (i.e., beta blockers, calcium channel blockers, etc.)",admissiondx_Tracheostomy,admissiondx_Transphenoidal surgery,"admissiondx_Trauma medical, other","admissiondx_Vascular medical, other","admissiondx_Vascular surgery, other",admissiondx_Weaning from mechanical ventilation (transfer from other unit or hospital only),admissiondx_nan
1,141765,129391,Female,87,Caucasian,59,91,157.5,23:36:00,-8,NaN,2015,19:20:00,2616,Home,Alive,Med-Surg ICU,23:44:00,Emergency Department,1,admit,46.5,45.0,13:14:00,2250,Step-Down Unit (SDU),Alive,002-1039,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,145427,132209,Male,61,Caucasian,68,103,177.8,23:48:00,-10,Emergency Depa

In [8]:
patient.drop(columns=['wardid', 'dischargeweight', 'hospitaladmittime24', 'hospitaladmitsource',
                      'unittype', 'unitadmittime24', 'unitadmitsource','unitdischargetime24', 'hospitaldischargetime24', 
                      'unitdischargeoffset', 
                      'hospitaldischargeoffset', 'hospitaldischargelocation', 'unitstaytype', 'unitdischargelocation', ], inplace=True)
patient

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,admissionheight,hospitaladmitoffset,hospitaldischargeyear,hospitaldischargestatus,unitvisitnumber,admissionweight,unitdischargestatus,uniquepid,mortality_at48h,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Anaphylaxis,admissiondx_Anemia,"admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)",admissiondx_Asthma,admissiondx_Atelectasis,"admissiondx_Bleeding, GI from esophageal varices/portal hypertension","admissiondx_Bleeding, GI-location unknown","admissiondx_Bleeding, lower GI","admissiondx_Bleeding, upper GI","admissiondx_Bleeding-other GI, surgery for","admissiondx_Bleeding-upper GI, surgery for",admissiondx_Burr hole placement,"admissiondx_CABG alone, coronary artery bypass grafting",admissiondx_CABG redo with other operation,admissiondx_CABG with aortic valve replacement,...,admissiondx_Peritonitis,"admissiondx_Pneumonia, aspiration","admissiondx_Pneumonia, bacterial","admissiondx_Pneumonia, other","admissiondx_Pneumonia, viral",admissiondx_Pneumothorax,admissiondx_Pre-eclampsia/eclampsia,admissiondx_Renal bleeding,"admissiondx_Renal failure, acute",admissiondx_Renal infection/abscess,admissiondx_Renal obstruction,"admissiondx_Respiratory - medical, other","admissiondx_Respiratory surgery, other","admissiondx_Restrictive lung disease (i.e., Sarcoidosis, pulmonary fibrosis)","admissiondx_Rhythm disturbance (atrial, supraventricular)",admissiondx_Rhythm disturbance (conduction defect),admissiondx_Rhythm disturbance (ventricular),admissiondx_Seizures (primary-no structural brain disease),"admissiondx_Sepsis, GI","admissiondx_Sepsis, cutaneous/soft tissue","admissiondx_Sepsis, other","admissiondx_Sepsis, pulmonary","admissiondx_Sepsis, renal/UTI (including bladder)","admissiondx_Sepsis, unknown","admissiondx_Shock, cardiogenic",admissiondx_Sickle cell crisis,"admissiondx_Skin surgery, other","admissiondx_Spinal cord surgery, other",admissiondx_Spinal/extremity trauma,admissiondx_Spinal/multiple trauma,admissiondx_Splenectomy,admissiondx_Stereotactic procedure,admissiondx_Subarachnoid hemorrhage/intracranial aneurysm,"admissiondx_Subarachnoid hemorrhage/intracranial aneurysm, surgery for","admissiondx_TURP, transurethral prostate resection for benign prostatic hypertrophy","admissiondx_Tamponade, pericardial","admissiondx_Thoracotomy for benign tumor (i.e. mediastinal chest wall mass, thymectomy)",admissiondx_Thoracotomy for lung cancer,admissiondx_Thoracotomy for pleural disease,admissiondx_Thrombectomy (with general anesthesia),"admissiondx_Thrombosis, vascular (deep vein)","admissiondx_Thrombus, arterial","admissiondx_Toxicity, drug (i.e., beta blockers, calcium channel blockers, etc.)",admissiondx_Tracheostomy,admissiondx_Transphenoidal surgery,"admissiondx_Trauma medical, other","admissiondx_Vascular medical, other","admissiondx_Vascular surgery, other",admissiondx_Weaning from mechanical ventilation (transfer from other unit or hospital only),admissiondx_nan
1,141765,129391,Female,87,Caucasian,59,157.5,-8,2015,Alive,1,46.5,Alive,002-1039,0,0,0,0,0,0,0,0,0,0,0

### Hospital table

The `hospital.csv.gz` table contains information about each hospital in the dataset. Each row corresponds to a hospital (`hospitalid`) and may include attributes such as region, hospital size, teaching status, and other structural characteristics.

This table is useful because the eICU dataset spans **many different hospitals**, each of which may have different:
* patient populations  
* clinical practices  
* resource availability  
* coding and data collection patterns  

By joining the `hospital` table with the `patient` table (using `hospitalid`), we can incorporate hospital-level features into our analysis.

This is especially important when thinking about **generalization and dataset shift**. For instance, a model trained on data from one set of hospitals may not perform well on another. 

In [9]:
hospital = pd.read_csv(f'{data_dir}/hospital.csv.gz')
hospital

,hospitalid,numbedscategory,teachingstatus,region
0,56,<100,f,Midwest
1,58,100 - 249,f,Midwest
2,59,<100,f,Midwest
3,60,<100,f,Midwest
4,61,<100,f,Midwest
...,...,...,...,...
181,445,100 - 249,f,Midwest
182,449,>= 500,t,Midwest
183,452,NaN,f,Midwest
184,458,>= 500,f,South


In [10]:
# Joint with the hospital table to get hospital-level features for each ICU visit
patient = patient.merge(hospital.drop(columns='teachingstatus').rename(
    columns={c: f"hospital_{c}" for c in hospital.columns if c != 'hospitalid'}), 
               on='hospitalid', how='left')
patient

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,admissionheight,hospitaladmitoffset,hospitaldischargeyear,hospitaldischargestatus,unitvisitnumber,admissionweight,unitdischargestatus,uniquepid,mortality_at48h,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Anaphylaxis,admissiondx_Anemia,"admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)",admissiondx_Asthma,admissiondx_Atelectasis,"admissiondx_Bleeding, GI from esophageal varices/portal hypertension","admissiondx_Bleeding, GI-location unknown","admissiondx_Bleeding, lower GI","admissiondx_Bleeding, upper GI","admissiondx_Bleeding-other GI, surgery for","admissiondx_Bleeding-upper GI, surgery for",admissiondx_Burr hole placement,"admissiondx_CABG alone, coronary artery bypass grafting",admissiondx_CABG redo with other operation,admissiondx_CABG with aortic valve replacement,...,"admissiondx_Pneumonia, bacterial","admissiondx_Pneumonia, other","admissiondx_Pneumonia, viral",admissiondx_Pneumothorax,admissiondx_Pre-eclampsia/eclampsia,admissiondx_Renal bleeding,"admissiondx_Renal failure, acute",admissiondx_Renal infection/abscess,admissiondx_Renal obstruction,"admissiondx_Respiratory - medical, other","admissiondx_Respiratory surgery, other","admissiondx_Restrictive lung disease (i.e., Sarcoidosis, pulmonary fibrosis)","admissiondx_Rhythm disturbance (atrial, supraventricular)",admissiondx_Rhythm disturbance (conduction defect),admissiondx_Rhythm disturbance (ventricular),admissiondx_Seizures (primary-no structural brain disease),"admissiondx_Sepsis, GI","admissiondx_Sepsis, cutaneous/soft tissue","admissiondx_Sepsis, other","admissiondx_Sepsis, pulmonary","admissiondx_Sepsis, renal/UTI (including bladder)","admissiondx_Sepsis, unknown","admissiondx_Shock, cardiogenic",admissiondx_Sickle cell crisis,"admissiondx_Skin surgery, other","admissiondx_Spinal cord surgery, other",admissiondx_Spinal/extremity trauma,admissiondx_Spinal/multiple trauma,admissiondx_Splenectomy,admissiondx_Stereotactic procedure,admissiondx_Subarachnoid hemorrhage/intracranial aneurysm,"admissiondx_Subarachnoid hemorrhage/intracranial aneurysm, surgery for","admissiondx_TURP, transurethral prostate resection for benign prostatic hypertrophy","admissiondx_Tamponade, pericardial","admissiondx_Thoracotomy for benign tumor (i.e. mediastinal chest wall mass, thymectomy)",admissiondx_Thoracotomy for lung cancer,admissiondx_Thoracotomy for pleural disease,admissiondx_Thrombectomy (with general anesthesia),"admissiondx_Thrombosis, vascular (deep vein)","admissiondx_Thrombus, arterial","admissiondx_Toxicity, drug (i.e., beta blockers, calcium channel blockers, etc.)",admissiondx_Tracheostomy,admissiondx_Transphenoidal surgery,"admissiondx_Trauma medical, other","admissiondx_Vascular medical, other","admissiondx_Vascular surgery, other",admissiondx_Weaning from mechanical ventilation (transfer from other unit or hospital only),admissiondx_nan,hospital_numbedscategory,hospital_region
0,141765,129391,Female,87,Caucasian,59,157.5,-8,2015,Alive,1,46.5,Alive,002-1039,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

In [11]:
### Aggregate multiple ICU stays into one row per hospitalization

# Recall:
# - patienthealthsystemstayid = one hospitalization
# - patientunitstayid         = one ICU stay within that hospitalization
#
# A single hospitalization can contain multiple ICU stays, including
# multiple ICU stays within our first 48-hour window.
#
# Since our prediction unit is the hospitalization (not the individual ICU stay),
# we want to collapse all ICU-stay-level rows into ONE row per
# patienthealthsystemstayid.

keep_cols = ['gender', 'age', 'ethnicity', 'hospitalid',
    'hospitaldischargeyear', 'mortality_at48h',
    'hospital_numbedscategory','hospital_region'
]

# These are the one-hot encoded admission diagnosis columns we created earlier.
admissiondx_cols = admissiondx_ohe.columns


patient_agg = patient.groupby('patienthealthsystemstayid').agg({
    **{c: 'first' for c in keep_cols},         # same within hospitalization so it does not matter whether we take the first, max, or min.
    **{c: 'max' for c in admissiondx_cols},    # keep diagnosis if it ever appears
    'admissionheight': 'mean',
    'admissionweight': 'mean',
}).reset_index()

# This prints the fraction of hospitalizations where mortality occurred
# within the 48-hour outcome window.
print(
    f"Mortality rate of patients that die between {hours_for_data}-{hours_mortality} hours: ",
    patient_agg[f'mortality_at{hours_mortality}h'].mean()
)

# Final hospitalization-level table
patient_agg

Mortality rate of patients that die between 24-48 hours:  0.033101045296167246


,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,hospitaldischargeyear,mortality_at48h,hospital_numbedscategory,hospital_region,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Anaphylaxis,admissiondx_Anemia,"admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)",admissiondx_Asthma,admissiondx_Atelectasis,"admissiondx_Bleeding, GI from esophageal varices/portal hypertension","admissiondx_Bleeding, GI-location unknown","admissiondx_Bleeding, lower GI","admissiondx_Bleeding, upper GI","admissiondx_Bleeding-other GI, surgery for","admissiondx_Bleeding-upper GI, surgery for",admissiondx_Burr hole placement,"admissiondx_CABG alone, coronary artery bypass grafting",admissiondx_CABG redo with other operation,admissiondx_CABG with aortic valve replacement,admissiondx_CABG with mitral valve repair,admissiondx_CABG with other operation,"admissiondx_CHF, congestive heart failure","admissiondx_CVA, cerebrovascular accident/stroke","admissiondx_Cancer, esophageal","admissiondx_Cancer, oral/sinus, surgery for",...,"admissiondx_Pneumonia, bacterial","admissiondx_Pneumonia, other","admissiondx_Pneumonia, viral",admissiondx_Pneumothorax,admissiondx_Pre-eclampsia/eclampsia,admissiondx_Renal bleeding,"admissiondx_Renal failure, acute",admissiondx_Renal infection/abscess,admissiondx_Renal obstruction,"admissiondx_Respiratory - medical, other","admissiondx_Respiratory surgery, other","admissiondx_Restrictive lung disease (i.e., Sarcoidosis, pulmonary fibrosis)","admissiondx_Rhythm disturbance (atrial, supraventricular)",admissiondx_Rhythm disturbance (conduction defect),admissiondx_Rhythm disturbance (ventricular),admissiondx_Seizures (primary-no structural brain disease),"admissiondx_Sepsis, GI","admissiondx_Sepsis, cutaneous/soft tissue","admissiondx_Sepsis, other","admissiondx_Sepsis, pulmonary","admissiondx_Sepsis, renal/UTI (including bladder)","admissiondx_Sepsis, unknown","admissiondx_Shock, cardiogenic",admissiondx_Sickle cell crisis,"admissiondx_Skin surgery, other","admissiondx_Spinal cord surgery, other",admissiondx_Spinal/extremity trauma,admissiondx_Spinal/multiple trauma,admissiondx_Splenectomy,admissiondx_Stereotactic procedure,admissiondx_Subarachnoid hemorrhage/intracranial aneurysm,"admissiondx_Subarachnoid hemorrhage/intracranial aneurysm, surgery for","admissiondx_TURP, transurethral prostate resection for benign prostatic hypertrophy","admissiondx_Tamponade, pericardial","admissiondx_Thoracotomy for benign tumor (i.e. mediastinal chest wall mass, thymectomy)",admissiondx_Thoracotomy for lung cancer,admissiondx_Thoracotomy for pleural disease,admissiondx_Thrombectomy (with general anesthesia),"admissiondx_Thrombosis, vascular (deep vein)","admissiondx_Thrombus, arterial","admissiondx_Toxicity, drug (i.e., beta blockers, calcium channel blockers, etc.)",admissiondx_Tracheostomy,admissiondx_Transphenoidal surgery,"admissiondx_Trauma medical, other","admissiondx_Vascular medical, other","admissiondx_Vascular surgery, other",admissiondx_Weaning from mechanical ventilation (transfer from other unit or hospital only),admissiondx_nan,admissi

In [12]:
# This will be useful for grouping types of features X together 
feature_sets = {'admissiondx': admissiondx_cols} 

### Diagnosis table


The `diagnosis.csv.gz` table contains structured diagnosis codes (e.g., ICD codes) that are assigned to a patient during their ICU stay. ICD codes (International Classification of Diseases) - for example, `I21` → acute myocardial infarction (heart attack) - are standardized codes used to represent diagnoses and medical conditions, mostly used for billing purposes (which inject their own form of biases... but that's a problem for another day). You can learn more about specific what the codes mean here: https://www.icd10data.com/ICD10CM/Codes. 

In `diagnosis.csv.gz` each row represents a diagnosis associated with a specific ICU visit (`patientunitstayid`), and includes timing information via offset columns.

This table is particularly useful for both **covariates (X)** and **outcomes (Y)**, depending on how we use timing. Again, to be rigorous researchers, it should be that covariates generally occur before outcomes we are trying to predict.

- **Covariates (X):**  
  We use diagnoses (along with labs and vitals) recorded in the first 24 hours of the hospitalization. These represent the information available early in a patient’s stay.

- **Outcomes (Y):**  
  We can define outcomes using events that occur *after* this window. For example, we may care about whether a patient is later diagnosed with myocardial infarction during the second day. Since we haven't decided specifically which outcome we want to predict yet, we will take a subset of them!   

In [13]:
diagnosis = pd.read_csv(f'{data_dir}/diagnosis.csv.gz')

# Keep only diagnosis rows for ICU stays that are present in our patient cohort <48 hours
# This removes diagnosis records for ICU stays we are not analyzing
diagnosis = diagnosis[diagnosis.patientunitstayid.isin(patient.patientunitstayid.unique())]

# Join in hospitaladmitoffset from the patient table
# We need this to align diagnosis events to time since hospital admission
diagnosis = diagnosis.merge(
    patient[['patientunitstayid', 'hospitaladmitoffset']],
    on='patientunitstayid',
    how='left'
)

# Compute timing of each diagnosis relative to hospital admission
# diagnosisoffset = minutes from ICU admission to diagnosis event
# hospitaladmitoffset = minutes from hospital admission to ICU admission
#   (typically stored as a negative number)
# Therefore: hospitaladmitoffset + diagnosisoffset
# gives the number of minutes after hospital admission that the diagnosis occurred
diagnosis['time_from_admit'] = -diagnosis.hospitaladmitoffset + diagnosis.diagnosisoffset

In [14]:
# Extract ICD10 codes from the diagnosis code column
# ---------------------------------------------------------
# The column contains unstructured strings that generally look like {icd9 code}, {equivalent icd10 code}
# In pracrtice, it may contain one or more diagnosis codes
#
# We use regex to extract codes that look like ICD10 codes:
#   - one capital letter 
#   - followed by 2 digits
#   - optionally followed by a decimal and 1-4 alphanumeric characters
#
# Examples:
#   I21
#   E11.9
#   N39.0
pattern = r"\b[A-TV-Z][0-9]{2}(?:\.[0-9A-Z]{1,4})?\b"

diagnosis['icd9code_pp'] = diagnosis.icd9code.astype(str).apply(lambda x: re.findall(pattern, x))

# explode(): If a row contains multiple diagnosis codes, create one row per code
diagnosis = diagnosis.explode("icd9code_pp").rename(
    columns={"icd9code_pp": "icd10"}
).drop(columns=[
    'icd9code',
    'diagnosispriority',
    'activeupondischarge',
    'diagnosisstring',
    'diagnosisid'
])
diagnosis


,patientunitstayid,diagnosisoffset,hospitaladmitoffset,time_from_admit,icd10
0,346380,5028,-2771,7799,I10
1,346380,685,-2771,3456,R41.82
2,346380,5035,-2771,7806,I95.9
3,346380,5035,-2771,7806,F20.9
4,346380,5028,-2771,7799,I26.99
...,...,...,...,...,...
19919,3139532,7081,-237,7318,J18.9
19920,3139532,9847,-237,10084,J18.9
19921,3139532,11275,-237,11512,J18.9
19922,3139532,5656,-237,5893,J18.9


In [15]:
# Split diagnosis events into "before 24h" (covariates X) and "after 24h" (possible outcomes Y, if we wish)

diagnosis_before24h = diagnosis[diagnosis.time_from_admit <= hours_for_data * 60]

# Diagnoses observed after 24 hours can optionally be used as OUTCOMES (Y)
# For example, we might ask:
#   "Can we predict whether a patient will later receive an MI diagnosis
#    based on their first-day data?"
diagnosis_after24h = diagnosis[diagnosis.time_from_admit > hours_for_data * 60]

# Inspect early diagnosis events
diagnosis_before24h

,patientunitstayid,diagnosisoffset,hospitaladmitoffset,time_from_admit,icd10
42,369273,65,-95,160,I50.9
43,369273,65,-95,160,NaN
44,369273,65,-95,160,I47.2
45,369273,65,-95,160,NaN
48,1981330,52,-13,65,K92.2
...,...,...,...,...,...
19910,2606115,1209,-90,1299,I50.9
19911,2606115,54,-90,144,J18.9
19912,2606115,1209,-90,1299,J18.9
19918,3139532,-4,-237,233,J18.9


In [16]:
# First we will handle covariates X 

# One-hot encode / multiple-hot encode diagnoses observed in the first 24 hours
diagnosis_ohe_before24h = pd.concat(
    [
        diagnosis_before24h[['patientunitstayid']],
        pd.get_dummies(diagnosis_before24h.icd10)
    ],
    axis=1
).astype(int).groupby('patientunitstayid').max().reset_index()

# Inspect ICU-stay-level diagnosis features from the first 24h
diagnosis_ohe_before24h

,patientunitstayid,A04.7,A09,A41.9,B17.1,B49,C06.9,C15.9,C16.9,C18.0,C18.9,C24.9,C25.9,C32.9,C34.90,C43.9,C50.919,C53.9,C55,C56.9,C61,C64.9,C67.9,C71.9,C73,C78.00,C78.6,C78.7,C79.31,C79.51,C80.0,C85.80,C90.00,C91.10,C92.10,C93.1,C94.6,D47.3,D57.1,D61.9,D62,D63.1,D63.8,D64.9,D68.32,D68.4,D68.69,D68.9,D69.6,D70.9,...,R57.0,R57.1,R57.9,R58,R65,R65.10,R65.2,R65.20,R65.21,R68.0,R73.9,R74.0,R78.81,S05,S05.1,S06.4,S06.5,S06.6,S06.9,S10.8,S12,S14.1,S22.00,S22.2,S22.4,S24.1,S27.30,S32.00,S32.30,S32.59,S36.0,S36.11,S36.50,S36.9,S73.00,T39.09,T39.1X,T40.5,T40.60,T42.4X,T43.01,T46.0X,T50.90,T51.0,T86.10,T86.11,T86.819,V08,V62.84,Z21
0,145427,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,151179,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,151867,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,151900,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,152954,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1516,3348292,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1517,3348409,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1518,3351297,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1519,3352231,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [17]:
# ---------------------------------------------------------
# Aggregate first-24h diagnosis features to the hospitalization level
# ---------------------------------------------------------
# diagnosis_ohe_before24h is currently one row per ICU stay
# But our prediction unit is one row per hospitalization
#
# So we:
# 1. Merge the ICU-stay diagnosis features onto the patient table
#    to recover patienthealthsystemstayid
# 2. Group by patienthealthsystemstayid
# 3. Take max() across ICU stays
#    -> if the diagnosis appears in any ICU stay within that hospitalization,
#       mark it as present
icd10_cols = diagnosis_ohe_before24h.columns.difference(['patientunitstayid'])

# Save the names of these diagnosis feature columns so we can reuse them later
feature_sets['icd10_before24h'] = icd10_cols

diagnosis_ohe_before24h_agg = patient.merge(
    diagnosis_ohe_before24h,
    on='patientunitstayid',
    how='left'
).groupby('patienthealthsystemstayid')[icd10_cols].max()

# View the most common ICD-10 diagnosis features in the first 24 hours
diagnosis_ohe_before24h_agg.mean().sort_values(ascending=False)

I10       0.145310
J18.9     0.112946
J96.00    0.107662
N17.9     0.089828
I48.0     0.085205
            ...   
I50.21    0.000661
J67.9     0.000661
J86.9     0.000661
J93.1     0.000661
I34.8     0.000661
Length: 363, dtype: float64

In [18]:
patient_agg = patient_agg.merge(diagnosis_ohe_before24h_agg, on= 'patienthealthsystemstayid', how='left')

In [19]:
# Next we will do the same for possible outcomes Y (diagnoses observed after 24h) 

# One-hot encode diagnoses observed after 24 hours
# These are not used as features here, but they may be useful as future labels
# or outcomes for prediction tasks
diagnosis_ohe_after24h = pd.concat(
    [
        diagnosis_after24h[['patientunitstayid']],
        pd.get_dummies(diagnosis_after24h.icd10)
    ],
    axis=1
).astype(int).groupby('patientunitstayid').max().add_prefix('label_').reset_index()

# Inspect ICU-stay-level diagnosis outcomes after 24h
diagnosis_ohe_after24h


,patientunitstayid,label_A04.7,label_A09,label_A41.9,label_B17.1,label_B25.0,label_C15.9,label_C16.9,label_C18.0,label_C18.9,label_C24.9,label_C25.9,label_C32.9,label_C34.90,label_C43.9,label_C50.919,label_C56.9,label_C61,label_C64.9,label_C67.9,label_C71.9,label_C78.00,label_C78.6,label_C78.7,label_C79.31,label_C85.80,label_C90.00,label_C91.10,label_C92.10,label_C93.1,label_C94.6,label_D45,label_D51.9,label_D57.1,label_D62,label_D63.1,label_D63.8,label_D64.9,label_D68.32,label_D68.69,label_D68.9,label_D69.6,label_D70.9,label_D72.825,label_D72.828,label_D72.829,label_E03.9,label_E05.90,label_E10.1,label_E10.65,...,label_R40.3,label_R41.82,label_R45.851,label_R50.9,label_R51,label_R55,label_R56.9,label_R57.0,label_R57.1,label_R57.8,label_R58,label_R65,label_R65.10,label_R65.2,label_R65.20,label_R65.21,label_R73.9,label_R74.0,label_R78.81,label_S06.4,label_S06.5,label_S06.6,label_S06.9,label_S10.8,label_S12,label_S14.1,label_S22.2,label_S22.4,label_S24.1,label_S27.30,label_S32.00,label_S32.30,label_S32.59,label_S36.0,label_S36.50,label_S36.89,label_S36.9,label_S73.00,label_T39.09,label_T39.1X,label_T40.60,label_T42.4X,label_T50.90,label_T51.0,label_T80.9,label_T86.10,label_T86.819,label_V08,label_V62.84,label_Z21
0,151179,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,151900,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,172015,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,187150,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,202294,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
634,3347501,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
635,3348292,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
636,3348409,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
637,3352230,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [20]:
# We can use this later for outcome labels Y of our choice 
icd10_cols = diagnosis_ohe_after24h.columns.difference(['patientunitstayid'])
feature_sets['icd10_after24h'] = icd10_cols
diagnosis_ohe_after24h_agg = patient.merge(diagnosis_ohe_after24h, on='patientunitstayid', how='left'
              ).groupby('patienthealthsystemstayid')[icd10_cols].max()
diagnosis_ohe_after24h_agg.mean().sort_values(ascending=False)

label_J96.00     0.197472
label_I10        0.181675
label_J18.9      0.164297
label_I48.0      0.146919
label_N17.9      0.126382
                   ...   
label_C50.919    0.001580
label_C56.9      0.001580
label_B25.0      0.001580
label_A09        0.001580
label_T80.9      0.001580
Length: 318, dtype: float64

In [21]:
patient_agg = patient_agg.merge(diagnosis_ohe_after24h_agg, on= 'patienthealthsystemstayid', how='left')

### Vitals table 

The eICU dataset contains two types of vital sign tables:

* `vitalPeriodic.csv.gz` → vitals recorded at regular intervals (e.g., heart rate, blood pressure from monitors)  
* `vitalAperiodic.csv.gz` → vitals recorded irregularly (e.g., nurse-charted measurements)  

Both tables capture time-series data about a patient’s physiological state. In this project, we summarize these time-series into features by aggregating values over time windows (e.g., first 24 hours).


In [22]:
# Load both vital sign tables
vital_periodic = pd.read_csv(f'{data_dir}/vitalPeriodic.csv.gz').drop(columns='vitalperiodicid')
vital_aperiodic = pd.read_csv(f'{data_dir}/vitalAperiodic.csv.gz').drop(columns='vitalaperiodicid')

In [23]:
# We will process both tables in the same way and store results here
vitals_dfs = []

# Define time bins (in hours → converted to minutes)
# We split the first 24 hours into two bins: [0–12), [12–24). You can choose to change these. 
hour_bins = [0, 12, 24]
time_bins = [h * 60 for h in hour_bins]  # convert hours → minutes
bin_labels = [f"bin{i+1}" for i in range(len(time_bins) - 1)]

# ---------------------------------------------------------
# Process each vitals table
# ---------------------------------------------------------
for df in [vital_periodic, vital_aperiodic]:

    # Keep only ICU stays in our cohort
    df = df[df.patientunitstayid.isin(patient.patientunitstayid.unique())]

    # Join patient info to compute time since hospital admission
    df = df.merge(
        patient[['patientunitstayid', 'patienthealthsystemstayid', 'hospitaladmitoffset']],
        on='patientunitstayid',
        how='left'
    )

    # Compute minutes since hospital admission
    df['time_from_admit'] = -df.hospitaladmitoffset + df.observationoffset

    # Keep only observations in the first 24 hours (feature window)
    df = df[df.time_from_admit <= hours_for_data * 60]

    # Assign each observation to a time bin
    df["time_bin"] = pd.cut(
        df["observationoffset"],
        bins=time_bins,
        labels=bin_labels,
        right=False,
        include_lowest=True
    )

    df = df.drop(columns=[
        'observationoffset',
        'hospitaladmitoffset',
        'time_from_admit',
        'patientunitstayid'
    ])

    # Aggregate vitals within each time bin
    # For each hospitalization and time bin:
    #   compute the mean of each vital sign
    df_expanded = df.groupby(
        ["patienthealthsystemstayid", "time_bin"]
    )[df.columns.difference(['patienthealthsystemstayid', 'time_bin'])] \
     .mean() \
     .unstack("time_bin")

    vitals_dfs.append(df_expanded)

# Split results for periodic and aperiodic vitals
vital_periodic_pp, vital_aperiodic_pp = vitals_dfs

# Check resulting feature shapes
vital_periodic_pp.shape, vital_aperiodic_pp.shape

((1572, 32), (1538, 20))

In [24]:
# Combine both periodic and aperiodic vitals into one table
vital = vital_periodic_pp.reset_index().merge(vital_aperiodic_pp.reset_index(), on='patienthealthsystemstayid', how='outer')
vital.columns = np.array(['patienthealthsystemstayid'] + [f"{col}_{bin}" for col, bin in vital.columns[1:]])
feature_sets['vitals'] = vital.columns[1:]

vital

/tmp/ipykernel_2887/1619381816.py:2: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  vital = vital_periodic_pp.reset_index().merge(vital_aperiodic_pp.reset_index(), on='patienthealthsystemstayid', how='outer')


,patienthealthsystemstayid,cvp_bin1,cvp_bin2,etco2_bin1,etco2_bin2,heartrate_bin1,heartrate_bin2,icp_bin1,icp_bin2,padiastolic_bin1,padiastolic_bin2,pamean_bin1,pamean_bin2,pasystolic_bin1,pasystolic_bin2,respiration_bin1,respiration_bin2,sao2_bin1,sao2_bin2,st1_bin1,st1_bin2,st2_bin1,st2_bin2,st3_bin1,st3_bin2,systemicdiastolic_bin1,systemicdiastolic_bin2,systemicmean_bin1,systemicmean_bin2,systemicsystolic_bin1,systemicsystolic_bin2,temperature_bin1,temperature_bin2,cardiacinput_bin1,cardiacinput_bin2,cardiacoutput_bin1,cardiacoutput_bin2,noninvasivediastolic_bin1,noninvasivediastolic_bin2,noninvasivemean_bin1,noninvasivemean_bin2,noninvasivesystolic_bin1,noninvasivesystolic_bin2,paop_bin1,paop_bin2,pvr_bin1,pvr_bin2,pvri_bin1,pvri_bin2,svr_bin1,svr_bin2,svri_bin1,svri_bin2
0,129391,NaN,NaN,NaN,NaN,77.902778,88.471831,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.782609,NaN,96.482517,96.890625,NaN,-0.152273,NaN,-0.750000,NaN,-0.613636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69.916667,70.545455,94.333333,96.090909,136.916667,148.909091,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,132209,NaN,NaN,NaN,NaN,64.008696,66.813008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.869565,21.363636,95.591304,97.545455,0.001739,-0.008943,-0.001304,0.220325,-0.025217,0.193089,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.227273,69.500000,82.545455,90.500000,116.045455,123.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,134042,NaN,NaN,NaN,NaN,90.042254,97.347222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.591549,27.548611,96.268116,95.472222,-0.177113,-0.354348,-0.651056,-0.636806,-0.450704,-0.261232,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.320000,83.190476,92.520000,117.523810,139.520000,176.190476,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,135537,NaN,NaN,NaN,NaN,68.896373,76.549451,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.294737,32.824176,92.265823,93.903846,-0.061702,0.002747,-0.009067,0.021429,0.039096,0.001648,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.200000,72.000000,82.800000,95.000000,111.200000,128.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,136669,118.764286,2.743056,NaN,NaN,101.443662,96.514085,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.359712,26.309859,96.901408,95.159722,-0.038732,-0.292553,-0.062676,-0.245390,-0.049648,-0.013475,NaN,NaN,NaN,NaN,NaN,NaN,30.706618,11.1,NaN,NaN,NaN,NaN,56.045714,56.750000,67.531429,69.044643,89.360000,92.008929,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1567,2738660,4.423077,247.216667,NaN,NaN,92.382353,69.641667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.372549,12.183333,97.078431,96.975000,-0.194444,0.012857,0.077778,0.052857,0.225926,-0.014286,54.212766,50.816667,71.255319,69.175,94.021277,96.991667,NaN,NaN,NaN,NaN,NaN,NaN,59.620000,52.750000,78.120000,72.166667,108.080000,99.166667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1568,2738759,NaN,NaN,NaN,NaN,89.875000,86.326389,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.500000,19.562500,96.601770,97.611111,0.000000,NaN,-0.042105,-0.134722,-0.060526,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66.285714,64.650000,94.628571,93.100000,140.714286,145.550000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1569,2741350,NaN,NaN,NaN,NaN,55.328571,51.746154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.950000,20.615385,99.507812,99.689922,5.750000,2.421875,16.428571,13.281250,4.642857,6.171875,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.680000,68.000000,89.320000,92.961538,126.200000,131.576923,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1570,2742186,NaN,NaN,NaN,NaN,62.608392,62.304348,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.020979,14.333333,95.452555,94.521739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.000000,71.500000,91.000000,92.000000,121.000000,129.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
patient_agg = patient_agg.merge(vital, on='patienthealthsystemstayid', how='left')
patient_agg

,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,hospitaldischargeyear,mortality_at48h,hospital_numbedscategory,hospital_region,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Anaphylaxis,admissiondx_Anemia,"admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)",admissiondx_Asthma,admissiondx_Atelectasis,"admissiondx_Bleeding, GI from esophageal varices/portal hypertension","admissiondx_Bleeding, GI-location unknown","admissiondx_Bleeding, lower GI","admissiondx_Bleeding, upper GI","admissiondx_Bleeding-other GI, surgery for","admissiondx_Bleeding-upper GI, surgery for",admissiondx_Burr hole placement,"admissiondx_CABG alone, coronary artery bypass grafting",admissiondx_CABG redo with other operation,admissiondx_CABG with aortic valve replacement,admissiondx_CABG with mitral valve repair,admissiondx_CABG with other operation,"admissiondx_CHF, congestive heart failure","admissiondx_CVA, cerebrovascular accident/stroke","admissiondx_Cancer, esophageal","admissiondx_Cancer, oral/sinus, surgery for",...,etco2_bin1,etco2_bin2,heartrate_bin1,heartrate_bin2,icp_bin1,icp_bin2,padiastolic_bin1,padiastolic_bin2,pamean_bin1,pamean_bin2,pasystolic_bin1,pasystolic_bin2,respiration_bin1,respiration_bin2,sao2_bin1,sao2_bin2,st1_bin1,st1_bin2,st2_bin1,st2_bin2,st3_bin1,st3_bin2,systemicdiastolic_bin1,systemicdiastolic_bin2,systemicmean_bin1,systemicmean_bin2,systemicsystolic_bin1,systemicsystolic_bin2,temperature_bin1,temperature_bin2,cardiacinput_bin1,cardiacinput_bin2,cardiacoutput_bin1,cardiacoutput_bin2,noninvasivediastolic_bin1,noninvasivediastolic_bin2,noninvasivemean_bin1,noninvasivemean_bin2,noninvasivesystolic_bin1,noninvasivesystolic_bin2,paop_bin1,paop_bin2,pvr_bin1,pvr_bin2,pvri_bin1,pvri_bin2,svr_bin1,svr_bin2,svri_bin1,svri_bin2
0,129391,Female,87,Caucasian,59,2015,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,NaN,NaN,77.902778,88.471831,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.782609,NaN,96.482517,96.890625,NaN,-0.152273,NaN,-0.750000,NaN,-0.613636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69.916667,70.545455,94.333333,96.090909,136.916667,148.909091,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,132209,Male,61,Caucasian,68,2014,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,NaN,NaN,64.008696,66.813008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.869565,21.363636,95.591304,97.545455,0.001739,-0.008943,-0.001304,0.220325,-0.025217,0.193089,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.227273,69.500000,82.545455,90.500000,116.045455,123.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,134042,Female,60,Hispanic,67,2015,0,None,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,NaN,NaN,90.042254,97.347222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.591549,27.548611,96.268116,95.472222,-0.177113,-0.354348,-0.651056,-0.636806,-0.450704,-0.261232,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.320000,83.190476,92.520000,117.523810,139.520000,176.190476,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,135537,Female,

### Lab table
The `lab.csv.gz` table contains laboratory test results collected during a patient’s ICU stay. Each row corresponds to a specific lab measurement (e.g., creatinine, white blood cell count) taken at a particular time.

In [26]:
lab = pd.read_csv(f'{data_dir}/lab.csv.gz')

# Keep only lab measurements for ICU stays in our patient cohort
lab = lab[lab.patientunitstayid.isin(patient.patientunitstayid.unique())]

# Clean lab names slightly
lab["labname"] = lab["labname"].str.replace("-", '')

# Align lab measurements to hospital admission time
lab = lab.merge(
    patient[['patientunitstayid', 'hospitaladmitoffset']],
    on='patientunitstayid',
    how='left'
)
# Compute minutes since hospital admission
# labresultoffset = minutes from ICU admission
# hospitaladmitoffset = minutes from hospital admission to ICU admission (typically negative)
lab['time_from_admit'] = -lab.hospitaladmitoffset + lab.labresultoffset

# Keep only lab measurements from the first 24 hours (feature window)
lab = lab[lab.time_from_admit <= hours_for_data * 60]

# Inspect filtered lab data
lab

,labid,patientunitstayid,labresultoffset,labtypeid,labname,labresult,labresulttext,labmeasurenamesystem,labmeasurenameinterface,labresultrevisedoffset,hospitaladmitoffset,time_from_admit
0,437880563,1754323,-647,3,Hct,38.30,38.3,%,%,-631,-695,48
1,437880572,1754323,-647,3,platelets x 1000,181.00,181,K/mcL,k/mm cu,-631,-695,48
2,437880560,1754323,-647,3,RBC,4.86,4.86,M/mcL,m/mm cu,-631,-695,48
3,437880570,1754323,-647,3,monos,8.70,8.7,%,%,-631,-695,48
4,437880571,1754323,-647,3,MCHC,30.40,30.4,g/dL,%,-631,-695,48
...,...,...,...,...,...,...,...,...,...,...,...,...
317023,641165127,2754778,37,1,CPK,283.00,283,Units/L,U/L,37,-16,53
317025,641165125,2754778,37,1,troponin I,5.92,5.920,ng/mL,ng/mL,37,-16,53
317026,667153606,2754778,37,3,MCV,92.00,92,fL,fL,37,-16,53
317027,673870691,2754778,236,4,bedside glucose,119.00,119,mg/dL,mg/dL,236,-16,252


In [27]:
lab.groupby(lab.labname).labmeasurenameinterface.value_counts()[:40] # ideally, these units should all be the same 

labname                   labmeasurenameinterface
24 h urine protein        mg/dL                        13
                          mg/day                        1
24 h urine urea nitrogen  mg/dL                         2
ALT (SGPT)                U/L                        1091
                          IU/L                        519
                          Units/L                     112
                          unit/L                       33
                          u/L                           8
                          u/l                           8
ANF/ANA                   titer                         1
AST (SGOT)                U/L                        1106
                          IU/L                        513
                          Units/L                     112
                          unit/L                       33
                          IU/l                         21
                          u/L                           8
                      

In [28]:
# Aggregate lab measurements into features
# For each (patientunitstayid, labname), compute the average lab value
lab_expanded = (
    lab.groupby(["patientunitstayid", "labname"])["labresult"]
        .mean()
        .unstack()   # convert lab names into columns
        .reset_index()
)
# Note: Missing values (NaN) mean that lab was not measured for that patient
lab_expanded

labname,patientunitstayid,24 h urine protein,24 h urine urea nitrogen,ALT (SGPT),ANF/ANA,AST (SGOT),Acetaminophen,BNP,BUN,Base Deficit,Base Excess,CPK,CPKMB,CPKMB INDEX,CRP,CRPhs,Carbamazepine,Carboxyhemoglobin,Clostridium difficile toxin A+B,Device,Digoxin,ESR,Fe,Fe/TIBC Ratio,Ferritin,FiO2,Gentamicin random,HCO3,HDL,Hct,Hgb,LDH,LDL,LPM O2,Lithium,MCH,MCHC,MCV,MPV,Methemoglobin,O2 Content,O2 Sat (%),Oxyhemoglobin,PEEP,PT,PT INR,PTT,PTT ratio,Peak Airway/Pressure,Phenobarbital,...,bicarbonate,calcium,chloride,cortisol,creatinine,direct bilirubin,eos,ethanol,fibrinogen,folate,free T4,glucose,glucose CSF,haptoglobin,ionized calcium,lactate,lipase,lymphs,magnesium,monos,myoglobin,pH,paCO2,paO2,phosphate,platelets x 1000,polys,potassium,prealbumin,prolactin,protein CSF,protein C,protein S,reticulocyte count,salicylate,serum ketones,serum osmolality,sodium,total bilirubin,total cholesterol,total protein,transferrin,triglycerides,troponin I,troponin T,uric acid,urinary creatinine,urinary osmolality,urinary sodium,urinary specific gravity
0,141765,NaN,NaN,29.0,NaN,28.0,NaN,478.0,31.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.650000,12.300000,NaN,NaN,NaN,NaN,29.850000,32.65,91.350000,NaN,NaN,NaN,NaN,NaN,NaN,19.4,1.80,NaN,NaN,NaN,NaN,...,22.000000,8.800000,103.000000,NaN,1.180000,NaN,0.5,NaN,NaN,NaN,NaN,76.0,NaN,NaN,NaN,NaN,NaN,14.500000,2.00,7.000000,NaN,NaN,NaN,NaN,NaN,215.0,78.000000,4.250000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,138.500000,0.80,NaN,7.40,NaN,NaN,0.020000,NaN,NaN,NaN,NaN,NaN,1.010
1,145427,NaN,NaN,29.0,NaN,18.0,NaN,NaN,16.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.300000,14.150000,NaN,NaN,NaN,NaN,30.800000,33.40,92.200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,28.000000,9.950000,106.000000,NaN,1.085000,NaN,0.0,NaN,NaN,NaN,NaN,128.5,NaN,NaN,NaN,NaN,194.0,6.500000,1.80,7.500000,NaN,NaN,NaN,NaN,NaN,207.5,83.500000,4.350000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.500000,0.80,NaN,6.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.015
2,147784,NaN,NaN,34.0,NaN,20.0,NaN,205.0,14.000000,NaN,9.75,67.0,1.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.5,NaN,40.50,NaN,44.150000,13.500000,NaN,NaN,NaN,NaN,31.250000,30.60,102.150000,NaN,NaN,NaN,NaN,NaN,NaN,10.1,1.00,28.000000,NaN,NaN,NaN,...,38.500000,8.200000,98.500000,NaN,0.640000,NaN,0.5,NaN,NaN,NaN,NaN,163.0,NaN,NaN,NaN,1.2,NaN,12.000000,2.00,5.000000,NaN,7.260000,74.0000,77.250,NaN,207.0,82.500000,4.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,141.000000,0.35,NaN,7.15,NaN,91.0,0.030000,NaN,NaN,NaN,NaN,NaN,NaN
3,149713,NaN,NaN,17.5,NaN,17.5,NaN,2025.0,10.500000,NaN,NaN,44.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.050000,12.550000,NaN,NaN,NaN,NaN,28.750000,33.00,87.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,23.000000,8.650000,99.000000,NaN,0.680000,NaN,2.0,NaN,NaN,NaN,NaN,98.0,NaN,NaN,NaN,NaN,NaN,13.000000,NaN,10.000000,NaN,NaN,NaN,NaN,NaN,253.0,74.000000,4.200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,133.500000,0.50,NaN,6.30,NaN,NaN,0.418000,NaN,NaN,NaN,NaN,NaN,NaN
4,151179,NaN,NaN,15.0,NaN,46.5,NaN,43.0,70.333333,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.0,NaN,10.00,NaN,40.266667,13.066667,NaN,NaN,2.0,NaN,29.900000,32.40,92.266667,NaN,NaN,NaN,NaN,NaN,NaN,24.0,2.20,38.000000,NaN,NaN,NaN,...,14.000000,6.833333,109.666667,33.1,2.736667,0.2,NaN,NaN,NaN,NaN,NaN,67.0,NaN,NaN,NaN,NaN,144.0,3.666667,NaN,8.333333,NaN,7.230000,26.0000,91.000,NaN,575.0,80.666667,4.966667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,140.000000,0.30,NaN,5.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [29]:
# Aggregate lab features to the hospitalization level

# Get list of lab feature columns (exclude ID column)
lab_cols = lab_expanded.columns.difference(['patientunitstayid'])

# Store feature names for later use
feature_sets['labs'] = lab_cols

# Merge lab features onto patient table to recover hospitalization IDs
# Then aggregate from ICU stay → hospitalization
lab_expanded_agg = (
    patient[['patientunitstayid', 'patienthealthsystemstayid']]
    .merge(lab_expanded, on='patientunitstayid', how='left')
    .groupby('patienthealthsystemstayid')[lab_cols]
    .mean()   # average across ICU stays within the same hospitalization
    .reset_index()
)
lab_expanded_agg

,patienthealthsystemstayid,24 h urine protein,24 h urine urea nitrogen,ALT (SGPT),ANF/ANA,AST (SGOT),Acetaminophen,BNP,BUN,Base Deficit,Base Excess,CPK,CPKMB,CPKMB INDEX,CRP,CRPhs,Carbamazepine,Carboxyhemoglobin,Clostridium difficile toxin A+B,Device,Digoxin,ESR,Fe,Fe/TIBC Ratio,Ferritin,FiO2,Gentamicin random,HCO3,HDL,Hct,Hgb,LDH,LDL,LPM O2,Lithium,MCH,MCHC,MCV,MPV,Methemoglobin,O2 Content,O2 Sat (%),Oxyhemoglobin,PEEP,PT,PT INR,PTT,PTT ratio,Peak Airway/Pressure,Phenobarbital,...,bicarbonate,calcium,chloride,cortisol,creatinine,direct bilirubin,eos,ethanol,fibrinogen,folate,free T4,glucose,glucose CSF,haptoglobin,ionized calcium,lactate,lipase,lymphs,magnesium,monos,myoglobin,pH,paCO2,paO2,phosphate,platelets x 1000,polys,potassium,prealbumin,prolactin,protein CSF,protein C,protein S,reticulocyte count,salicylate,serum ketones,serum osmolality,sodium,total bilirubin,total cholesterol,total protein,transferrin,triglycerides,troponin I,troponin T,uric acid,urinary creatinine,urinary osmolality,urinary sodium,urinary specific gravity
0,129391,NaN,NaN,29.0,NaN,28.0,NaN,478.0,31.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.650000,12.300000,NaN,NaN,NaN,NaN,29.850000,32.65,91.350000,NaN,NaN,NaN,NaN,NaN,NaN,19.4,1.80,NaN,NaN,NaN,NaN,...,22.000000,8.800000,103.000000,NaN,1.180000,NaN,0.5,NaN,NaN,NaN,NaN,76.0,NaN,NaN,NaN,NaN,NaN,14.500000,2.00,7.000000,NaN,NaN,NaN,NaN,NaN,215.0,78.000000,4.250000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,138.500000,0.80,NaN,7.40,NaN,NaN,0.020000,NaN,NaN,NaN,NaN,NaN,1.010
1,132209,NaN,NaN,29.0,NaN,18.0,NaN,NaN,16.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.300000,14.150000,NaN,NaN,NaN,NaN,30.800000,33.40,92.200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,28.000000,9.950000,106.000000,NaN,1.085000,NaN,0.0,NaN,NaN,NaN,NaN,128.5,NaN,NaN,NaN,NaN,194.0,6.500000,1.80,7.500000,NaN,NaN,NaN,NaN,NaN,207.5,83.500000,4.350000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.500000,0.80,NaN,6.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.015
2,134042,NaN,NaN,34.0,NaN,20.0,NaN,205.0,14.000000,NaN,9.750,67.0,1.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.5,NaN,40.500,NaN,44.150000,13.500000,NaN,NaN,NaN,NaN,31.250000,30.60,102.150000,NaN,NaN,NaN,NaN,NaN,NaN,10.1,1.00,28.000000,NaN,NaN,NaN,...,38.500000,8.200000,98.500000,NaN,0.640000,NaN,0.5,NaN,NaN,NaN,NaN,163.0,NaN,NaN,NaN,1.2,NaN,12.000000,2.00,5.000000,NaN,7.260000,74.00000,77.2500,NaN,207.0,82.500000,4.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,141.000000,0.35,NaN,7.15,NaN,91.0,0.030000,NaN,NaN,NaN,NaN,NaN,NaN
3,135537,NaN,NaN,17.5,NaN,17.5,NaN,2025.0,10.500000,NaN,NaN,44.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.050000,12.550000,NaN,NaN,NaN,NaN,28.750000,33.00,87.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,23.000000,8.650000,99.000000,NaN,0.680000,NaN,2.0,NaN,NaN,NaN,NaN,98.0,NaN,NaN,NaN,NaN,NaN,13.000000,NaN,10.000000,NaN,NaN,NaN,NaN,NaN,253.0,74.000000,4.200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,133.500000,0.50,NaN,6.30,NaN,NaN,0.418000,NaN,NaN,NaN,NaN,NaN,NaN
4,136669,NaN,NaN,15.0,NaN,46.5,NaN,43.0,70.333333,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.0,NaN,10.000,NaN,40.266667,13.066667,NaN,NaN,2.0,NaN,29.900000,32.40,92.266667,NaN,NaN,NaN,NaN,NaN,NaN,24.0,2.20,38.000000,NaN,NaN,NaN,...,14.000000,6.833333,109.666667,33.1,2.736667,0.2,NaN,NaN,NaN,NaN,NaN,67.0,NaN,NaN,NaN,NaN,144.0,3.666667,NaN,8.333333,NaN,7.230000,26.00000,91.0000,NaN,575.0,80.666667,4.966667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,140.000000,0.30,NaN,5.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [30]:
# Merge lab features into the main aggregated dataset
patient_agg = patient_agg.merge(
    lab_expanded_agg,
    on='patienthealthsystemstayid',
    how='left'
)

# Final dataset now includes lab features
patient_agg

,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,hospitaldischargeyear,mortality_at48h,hospital_numbedscategory,hospital_region,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Anaphylaxis,admissiondx_Anemia,"admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)",admissiondx_Asthma,admissiondx_Atelectasis,"admissiondx_Bleeding, GI from esophageal varices/portal hypertension","admissiondx_Bleeding, GI-location unknown","admissiondx_Bleeding, lower GI","admissiondx_Bleeding, upper GI","admissiondx_Bleeding-other GI, surgery for","admissiondx_Bleeding-upper GI, surgery for",admissiondx_Burr hole placement,"admissiondx_CABG alone, coronary artery bypass grafting",admissiondx_CABG redo with other operation,admissiondx_CABG with aortic valve replacement,admissiondx_CABG with mitral valve repair,admissiondx_CABG with other operation,"admissiondx_CHF, congestive heart failure","admissiondx_CVA, cerebrovascular accident/stroke","admissiondx_Cancer, esophageal","admissiondx_Cancer, oral/sinus, surgery for",...,bicarbonate,calcium,chloride,cortisol,creatinine,direct bilirubin,eos,ethanol,fibrinogen,folate,free T4,glucose,glucose CSF,haptoglobin,ionized calcium,lactate,lipase,lymphs,magnesium,monos,myoglobin,pH,paCO2,paO2,phosphate,platelets x 1000,polys,potassium,prealbumin,prolactin,protein CSF,protein C,protein S,reticulocyte count,salicylate,serum ketones,serum osmolality,sodium,total bilirubin,total cholesterol,total protein,transferrin,triglycerides,troponin I,troponin T,uric acid,urinary creatinine,urinary osmolality,urinary sodium,urinary specific gravity
0,129391,Female,87,Caucasian,59,2015,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,22.000000,8.800000,103.000000,NaN,1.180000,NaN,0.5,NaN,NaN,NaN,NaN,76.0,NaN,NaN,NaN,NaN,NaN,14.500000,2.00,7.000000,NaN,NaN,NaN,NaN,NaN,215.0,78.000000,4.250000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,138.500000,0.80,NaN,7.40,NaN,NaN,0.020000,NaN,NaN,NaN,NaN,NaN,1.010
1,132209,Male,61,Caucasian,68,2014,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,28.000000,9.950000,106.000000,NaN,1.085000,NaN,0.0,NaN,NaN,NaN,NaN,128.5,NaN,NaN,NaN,NaN,194.0,6.500000,1.80,7.500000,NaN,NaN,NaN,NaN,NaN,207.5,83.500000,4.350000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.500000,0.80,NaN,6.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.015
2,134042,Female,60,Hispanic,67,2015,0,None,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,38.500000,8.200000,98.500000,NaN,0.640000,NaN,0.5,NaN,NaN,NaN,NaN,163.0,NaN,NaN,NaN,1.2,NaN,12.000000,2.00,5.000000,NaN,7.260000,74.00000,77.2500,NaN,207.0,82.500000,4.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,141.000000,0.35,NaN,7.15,NaN,91.0,0.030000,NaN,NaN,NaN,NaN,NaN,NaN
3,135537,Female,> 89,Caucasian,60,2015,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,23.000000,8.650000,99.000000,NaN,0.680000,NaN,2.0,NaN,NaN,NaN,NaN,98.0,NaN,NaN,NaN,NaN,NaN,13.000000,NaN,10.000000,NaN,NaN,NaN,NaN,NaN,253.0,74.000000,4.20

In [31]:
os.getcwd()

'/home/hatch/workspace/cs197-project3/work'

### Save your data for future usage 

In [32]:
import pickle as pk 
patient_agg.to_parquet('../data/clean_dataset.parquet', compression='gzip') # 'gzip' compression is efficient
pk.dump(feature_sets, open(f'../data/feature_names.pkl', 'wb'))

---


# 3. Section Starter: Now it's your turn! 
We have our data in `patient_agg`. Next, we'll walk through possible other tables we could have used, and explore what the data in patient_agg looks like. 

We'll end with a simple ML prediction task. 

### 1. Exploring the medication table 

In [33]:
# TODO: open the medication table and list the most common medications ordered
# BONUS: can you list the most common medications ordered in the first 24 hours of hospitalization ?
# HINT: drugorderoffset + hospitaladmitoffset (which you can get from joining on the patient table) = minutes into the hospital stay the medication was ordered 

### 2. Analyze demographic features of patient_agg

In [34]:
## TODO: Using the patient_agg table, explore and visualize the counts of the patient features, 
#            such as age, gender, ethnicity, height and weight 

### 3. Analyze hospital information 

In [35]:
## TODO 1 : Explore how many patients are recorded for each hospitalid, hospital region.  
# Hint: patient_agg already has the 'hospitalid' and 'hospital_region' column. 
# TODO 2: Why might hospital region or hospital id be an important variable to know in ML models? 

### 4. Analyzing missingness 
As we've alluded to, real-world data is imperfect. For some patients, there are features we may have no information for. b

In [36]:
## TODO 1: Examine which lab variables have the most missing data in patient_agg. 
# Hint: feature_sets['labs'] lists all the lab features in patient_agg
# Hint: .isna() is useful in pandas to see which values are missing (NaN)

In [37]:
## TODO 2: Suppose you want to use labe features in an ML model. How might you handle the missing values? 

### 5. Picking an outcome variable Y to predict 
We might care about predicting several different outcome variables. So far, we've collected a few outcome variables:
- all the diagnoses ICD10 codes made AFTER 24 hours
- mortality at 48 hours 

In [38]:
patient_agg[feature_sets['icd10_after24h']].mean().sort_values(ascending=False)[:20]

label_J96.00     0.197472
label_I10        0.181675
label_J18.9      0.164297
label_I48.0      0.146919
label_N17.9      0.126382
label_A41.9      0.109005
label_I50.9      0.105845
label_J44.9      0.085308
label_J96.91     0.082148
label_R41.82     0.077409
label_I95.9      0.077409
label_R73.9      0.071090
label_K92.2      0.067930
label_J44.1      0.061611
label_D72.829    0.058452
label_E78.5      0.055292
label_R65.21     0.053712
label_R65.20     0.052133
label_F10.239    0.047393
label_R65.2      0.044234
dtype: float64

In [39]:
Y_label = 'mortality_at48h' # TODO: change if you want ! 
# TODO: Select an outcome Y as 
# either 'mortality_at48h' or pick an ICD10 code of your choice 'label_...' (using https://www.icd10data.com/ICD10CM/Codes) 


### 5. A simple ML prediction task
Next, we will walk through a simple prediciton task using an outcome Y of your choice, using a basic logistic regression model. 

In [40]:
covariate_names = ['age', 
          'heartrate_bin1', 'heartrate_bin2', # Heart rate binned over the first 24 hours
          'respiration_bin1', 'respiration_bin2', # Respiratory rate binned over the first 24 hours
          'admissiondx_Infarction, acute myocardial (MI)', # Admitted to the ICU with MI 
          'admissiondx_Sepsis, pulmonary', # Admitted to the ICU with sepsis 
          'admissiondx_CVA, cerebrovascular accident/stroke', # Admitted to the ICU with stroke 
          'admissiondx_CHF, congestive heart failure',# Admitted to the ICU with CHF 
          ] # You can change this to be anything <24h in patient_agg! 
X = patient_agg[covariate_names].copy()
Y = patient_agg[Y_label].copy().fillna(0) # If you picked an ICD10 code as your Y, then we assume that missing values mean the patient didn't have that ICD10 code observed 
X

,age,heartrate_bin1,heartrate_bin2,respiration_bin1,respiration_bin2,"admissiondx_Infarction, acute myocardial (MI)","admissiondx_Sepsis, pulmonary","admissiondx_CVA, cerebrovascular accident/stroke","admissiondx_CHF, congestive heart failure"
0,87,77.902778,88.471831,24.782609,NaN,0,0,0,0
1,61,64.008696,66.813008,15.869565,21.363636,0,0,0,0
2,60,90.042254,97.347222,21.591549,27.548611,0,0,0,0
3,> 89,68.896373,76.549451,30.294737,32.824176,1,0,0,0
4,59,101.443662,96.514085,28.359712,26.309859,0,0,0,0
...,...,...,...,...,...,...,...,...,...
1717,71,89.875000,86.326389,23.500000,19.562500,0,0,0,0
1718,52,55.328571,51.746154,22.950000,20.615385,0,0,0,0
1719,41,62.608392,62.304348,22.020979,14.333333,1,0,0,0
1720,72,123.433824,120.597222,37.603053,43.361111,0,0,0,0


In [41]:
# This distinction of numerical (continuous), binary, and categorical columns may be useful for you 
num_cols = feature_sets['labs'].tolist() + feature_sets['vitals'].tolist() + ['age', 'admissionheight', 'admissionweight']
bin_cols = feature_sets['icd10_before24h'].tolist() + feature_sets['admissiondx'].tolist()
cat_cols = ['hospital_region', 'ethnicity', 'gender', 'hospital_numbedscategory', 'hospitaldischargeyear', 'hospitalid']

# For privacy I think, the dataset sets the age of everyone over 89 as the string '>89'. We set back to a continuous number 
if 'age' in X.columns: 
    X.loc[X.age == '> 89', 'age'] = 90
    X['age'] = X['age'].astype(float)


In [42]:
### You'll find you cant just pass in X,Y into an ML model because of the missing values and potential categorical variables 
# TODO: Figure out how to handle the missing values and categorical variables (i.e., if you use `ethnicity` as a covariate in X)
# Hint: Sklearn has a few useful functions: from sklearn.preprocessing import OneHotEncoder
# or from sklearn.impute import SimpleImputer. We can choose median or mean imputation for now, 
# but we will explore in Project 2 why this might actually be bad sometimes !! 

In [43]:
### TODO: Use sklearn to build a simple logistic regression model to predict Y given X, using train_test_split to 
## separate into training and test sets. 
## Output the AUROC and AUPRC score on the test set. 
## Hint: I recommend using StandardScaler for normalizing continuous features 

In [44]:
## TODO: How well does your model do? Why might it be doing poorly / well? 

---

🎉 You did it! Congrats! 